In [2]:
from pathlib import Path

import altair as alt
import pandas as pd
from ecostyles import EcoStyles

styles = EcoStyles(); styles.register_and_enable_theme()

# ---------- data path: ../data first, else search up the folder tree ----------
FILENAME = "plot1+2_debt-to-gdp_g20.csv"
PATH = Path("../data") / FILENAME
if not PATH.exists():
    PATH = next(
        p for parent in [Path.cwd(), *list(Path.cwd().parents)[:4]]
        for p in parent.rglob(FILENAME)
    )
print("Using:", PATH.resolve())

# ---------- settings ----------
YEAR = 2025                    # latest year with full data; 2026+ in the file are projections
HIGHLIGHT = "Germany"
G7_ONLY = False                # True = G7 countries only
G7 = ["Canada", "France", "Germany", "Italy", "Japan", "United Kingdom", "United States"]
GREY = "#676A86"
MUTED = "#B8BBC9"              # non-highlighted bars

# ---------- data ----------
raw = pd.read_csv(PATH, na_values="no data")
raw = raw.loc[:, ~raw.columns.str.startswith("Unnamed")]       # blank second column

plot_df = (
    raw.loc[raw["Year"] == YEAR]
    .drop(columns="Year")
    .melt(var_name="country", value_name="debt")
    .dropna()
)
plot_df["country"] = (
    plot_df["country"]
    .str.replace(", People's Republic of", "", regex=False)
    .str.replace(", Republic of", "", regex=False)
    .replace({"Korea": "South Korea", "Russian Federation": "Russia"})
)
if G7_ONLY:
    plot_df = plot_df[plot_df["country"].isin(G7)]
plot_df["highlight"] = plot_df["country"].where(plot_df["country"] == HIGHLIGHT, "Other")

# ---------- chart ----------
y = alt.Y("country:N", sort="-x", title=None, scale=alt.Scale(paddingInner=0.3))
x = alt.X("debt:Q", axis=None, scale=alt.Scale(domain=[0, plot_df["debt"].max() * 1.1]))

bars = alt.Chart(plot_df).mark_bar().encode(
    y=y, x=x,
    color=alt.condition(
        alt.datum.highlight == HIGHLIGHT,
        alt.Color("highlight:N", legend=None),    # highlight takes the theme colour
        alt.value(MUTED),
    ),
)
labels = alt.Chart(plot_df).mark_text(
    align="left", baseline="middle", dx=4, fontSize=10, color=GREY,
).encode(y=y, x="debt:Q", text=alt.Text("debt:Q", format=".0f"))

caption = alt.Title(
    text="Source: IMF World Economic Outlook",
    subtitle=[f"General government gross debt, % of GDP, {YEAR}."],
    orient="bottom", anchor="start",
    fontSize=11, subtitleFontSize=10,
    color=GREY, subtitleColor=GREY, dy=12)

chart = (
    (bars + labels)
    .properties(width=400, height=300)
    .configure(background="white", font="Circular Std")
    .configure_view(fill="transparent", stroke="transparent"))

styles.save(chart, path=".", name="g20_debt_to_gdp", svg=True)
chart

Using: /Users/g.rushworth/Documents/GitHub/RADataHub/Debt 2026/Peterson Blogs/Roger Schneider/data/plot1+2_debt-to-gdp_g20.csv


alt.LayerChart(...)